In [ ]:
# Plot all points

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv("../data/zinc_dft/properties.csv")
df.plot.scatter(x=df.columns[0], y=df.columns[1], s=0.2)

In [ ]:
# Initial points

from blox2 import split_df_by_n_rows

df_obs, df_unchecked = split_df_by_n_rows(df, 10)
df_obs.plot.scatter(x=df_obs.columns[0], y=df_obs.columns[1], s=20)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler

def stein_score_on_grid(Y_obs: np.ndarray, sigma: float,
    x_min=0.0, x_max=1500.0, x_step=10.0,
    y_min=0.0, y_max=1.75, y_step=0.01, Y_for_normalization: np.ndarray=None):
    """
    Returns:
        xs: (nx,) grid x coordinates
        ys: (ny,) grid y coordinates
        Z:  (ny, nx) score values on the grid
    """
    sigma2 = float(sigma) ** 2
    d = 2
    
    Y = np.asarray(Y_obs, dtype=float)
    if Y.ndim != 2 or Y.shape[1] != 2:
        raise ValueError(f"Y_obs must be shape (n,2); got {Y.shape}")
    if not np.isfinite(sigma) or sigma <= 0:
        raise ValueError(f"sigma must be positive finite; got {sigma}")

    xs = np.arange(x_min, x_max + 0.5 * x_step, x_step)
    ys = np.arange(y_min, y_max + 0.5 * y_step, y_step)

    Xg, Yg = np.meshgrid(xs, ys, indexing="xy") # (ny, nx)
    G = np.stack([Xg.ravel(), Yg.ravel()], axis=1) # (m, 2)

    # standard scaling with observed points
    scaler = StandardScaler()
    if Y_for_normalization is not None:
        scaler.fit(Y_for_normalization)
    else:
        scaler.fit(Y)
    Y = scaler.transform(Y)
    G = scaler.transform(G)

    diff = Y[None, :, :] - G[:, None, :] # (m, n_obs, 2)
    dist = np.sum(diff * diff, axis=2) # (m, n_obs)

    scores = np.sum((dist - d * sigma2) * np.exp(-dist / (2.0 * sigma2)), axis=1) # (m,)
    Z = scores.reshape(len(ys), len(xs)) # (ny, nx)

    return xs, ys, Z

# Plot

sigma = 0.2
x_min, x_max, x_step = 0, 1500, 10
y_min, y_max, y_step = 0.0, 1.5, 0.01
# x_min, x_max, x_step = 200, 400, 1
# y_min, y_max, y_step = 0.0, 0.2, 0.001

Y_obs = df_obs[["Absorption wavelength (nm)", "Intensity"]].to_numpy(dtype=float)
Y_for_normalization = df[["Absorption wavelength (nm)", "Intensity"]].to_numpy(dtype=float)
# xs, ys, Z = stein_score_on_grid(Y_obs, sigma=sigma, x_min=x_min, x_max=x_max, x_step=x_step, y_min=y_min, y_max=y_max, y_step=y_step, Y_for_normalization=None)
xs, ys, Z = stein_score_on_grid(Y_obs, sigma=sigma, x_min=x_min, x_max=x_max, x_step=x_step, y_min=y_min, y_max=y_max, y_step=y_step, Y_for_normalization=Y_for_normalization)

plt.figure(figsize=(6.5, 5))
im = plt.imshow(Z, origin="lower", aspect="auto", extent=[xs.min(), xs.max(), ys.min(), ys.max()])
plt.colorbar(im, label="Stein novelty score")

plt.scatter(Y_obs[:, 0], Y_obs[:, 1], s=12, marker="o", linewidths=0, label="observed points")
plt.xlabel("Absorption wavelength (nm)")
plt.ylabel("Intensity")
plt.title(f"Stein novelty score (σ={sigma:g})")
plt.legend(loc="upper right")
plt.tight_layout()
plt.show()